<a href="https://colab.research.google.com/github/Prashkov1ch/python-ai-Prashkovich-Anna/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_Lab02_Economists_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 2 (экономисты): NumPy-массивы и базовая аналитика

**Ограничения:** используйте `numpy` и `re`.  
(Данные читаются из Excel в подготовительной ячейке; далее работаем с NumPy.)

**Цель:** собрать массивы показателей (выручка, чистая прибыль, активы, капитал) и отработать:
- универсальные функции (ufunc),
- сравнение и маски,
- булева логика,
- индексация и сортировка,
- агрегирование.

In [4]:
import re
from openpyxl import load_workbook
import numpy as np
import os
from google.colab import drive

# 1. Монтируем диск (если уже смонтирован, просто игнорируем ошибку)
try:
    drive.mount('/content/drive')
except:
    pass

PATH = '/content/drive/MyDrive/цифровая кафедра/'  # Проверь, что имя папки верное!
os.chdir(PATH)

FILES = ['Копия 1 Атомэнергопром.xlsx', 'Копия 2 Аэрофлот.xlsx', 'Копия 3 Газпром_петрозаводск.xlsx',
         'Копия 4 Лукойл.xlsx', 'Копия 5 Роснефть.xlsx', 'Копия 6 Самолет.xlsx',
         'Копия 7 Славмо.xlsx', 'Копия 8 Строительная_компания_Век.xlsx',
         'Копия 9 ТГК_1.xlsx', 'Копия 10 ТНС_ЭНЭРГО_Карелия.xlsx']

# --- ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ ДЛЯ ОЧИСТКИ ЧИСЕЛ ---
def clean_number(value):
    """
    Превращает значение из Excel в число float.
    Удаляет пробелы из строк вида '1 000 000'.
    Если значение пустое — возвращает np.nan.
    """
    if value is None:
        return np.nan
    # Превращаем в строку и убираем все пробелы
    s = str(value).replace(' ', '')
    try:
        return float(s)
    except:
        return np.nan

def load_org_name(xlsx_name: str):
    wb = load_workbook(xlsx_name, data_only=True)
    ws = wb["Сведения об организации"]
    return str(ws.cell(row=6, column=13).value)

def parse_financial(xlsx_name: str):
    wb = load_workbook(xlsx_name, data_only=True)
    ws = wb["Отчет о финансовых результатах"]
    # колонки значений: 2023 -> 21, 2022 -> 27
    res = {}
    for r in range(6, 200):
        code = ws.cell(r, 16).value
        if code is None:
            continue
        code = str(code).strip()
        if not re.fullmatch(r"\d{4}", code):
            continue
        v23 = ws.cell(r, 21).value
        v22 = ws.cell(r, 27).value
        # ИСПРАВЛЕНИЕ: используем clean_number для очистки данных
        res[code] = (clean_number(v22), clean_number(v23))
    return res

def parse_balance(xlsx_name: str):
    wb = load_workbook(xlsx_name, data_only=True)
    ws = wb["Бухгалтерский баланс"]
    # значения: 2023 -> 17, 2022 -> 23
    res = {}
    for r in range(6, 400):
        code = ws.cell(r, 14).value
        if code is None:
            continue
        code = str(code).strip()
        if not re.fullmatch(r"\d{4}", code):
            continue
        v23 = ws.cell(r, 17).value
        v22 = ws.cell(r, 23).value
        # ИСПРАВЛЕНИЕ: используем clean_number для очистки данных
        res[code] = (clean_number(v22), clean_number(v23))
    return res

# Загрузка данных
names = [load_org_name(fn) for fn in FILES]

revenue = []
net_profit = []
assets = []
equity = []

for fn in FILES:
    fin = parse_financial(fn)
    bal = parse_balance(fn)
    revenue.append(fin.get("2110", (np.nan, np.nan)))
    net_profit.append(fin.get("2400", (np.nan, np.nan)))
    assets.append(bal.get("1600", (np.nan, np.nan)))
    equity.append(bal.get("1300", (np.nan, np.nan)))

# Теперь массивы создадутся без ошибок, так как данные уже очищены
revenue = np.array(revenue, dtype=float)       # (n,2): [2022,2023]
net_profit = np.array(net_profit, dtype=float) # (n,2)
assets = np.array(assets, dtype=float)         # (n,2)
equity = np.array(equity, dtype=float)         # (n,2)

years = np.array([2022, 2023])

print("Shapes: ", revenue.shape, net_profit.shape, assets.shape, equity.shape)
print("Companies: ", len(names))
print("✅ Данные успешно загружены и очищены!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shapes:  (10, 2) (10, 2) (10, 2) (10, 2)
Companies:  10
✅ Данные успешно загружены и очищены!


### Задание 1
Проверьте форму и типы массивов `revenue`, `net_profit`, `assets`, `equity`.  
Выведите `dtype` и `shape` для каждого массива.

In [5]:
# Задание 1: Проверка формы и типов массивов

print("=" * 50)
print("Задание 1: Проверка формы и типов массивов")
print("=" * 50)

# 1. Проверяем массив выручки (revenue)
print("\n1. Выручка (revenue):")
print(f"   Форма (shape): {revenue.shape}")
print(f"   Тип данных (dtype): {revenue.dtype}")

# 2. Проверяем массив чистой прибыли (net_profit)
print("\n2. Чистая прибыль (net_profit):")
print(f"   Форма (shape): {net_profit.shape}")
print(f"   Тип данных (dtype): {net_profit.dtype}")

# 3. Проверяем массив активов (assets)
print("\n3. Активы (assets):")
print(f"   Форма (shape): {assets.shape}")
print(f"   Тип данных (dtype): {assets.dtype}")

# 4. Проверяем массив капитала (equity)
print("\n4. Капитал (equity):")
print(f"   Форма (shape): {equity.shape}")
print(f"   Тип данных (dtype): {equity.dtype}")

# 5. Выводим общую информацию
print("\n" + "=" * 50)
print("Общая информация:")
print("=" * 50)
print(f"Количество компаний: {len(names)}")
print(f"Количество лет: {len(years)} ({years[0]} и {years[1]})")
print(f"Размерность всех массивов: (10 компаний, 2 года)")

Задание 1: Проверка формы и типов массивов

1. Выручка (revenue):
   Форма (shape): (10, 2)
   Тип данных (dtype): float64

2. Чистая прибыль (net_profit):
   Форма (shape): (10, 2)
   Тип данных (dtype): float64

3. Активы (assets):
   Форма (shape): (10, 2)
   Тип данных (dtype): float64

4. Капитал (equity):
   Форма (shape): (10, 2)
   Тип данных (dtype): float64

Общая информация:
Количество компаний: 10
Количество лет: 2 (2022 и 2023)
Размерность всех массивов: (10 компаний, 2 года)


### Задание 2
Постройте массив роста выручки `rev_growth = revenue[:,1] - revenue[:,0]`.  
Выведите 3 компании с наибольшим ростом (используйте `argsort`).

In [6]:
# Задание 2: Расчёт роста выручки и поиск топ-3 компаний

print("=" * 60)
print("Задание 2: Рост выручки и топ-3 компании")
print("=" * 60)

# 1. Рассчитываем рост выручки: 2023 год (индекс 1) минус 2022 год (индекс 0)
# revenue[:,1] — все строки, столбец 1 (2023 год)
# revenue[:,0] — все строки, столбец 0 (2022 год)
rev_growth = revenue[:, 1] - revenue[:, 0]

# 2. Выводим массив роста для проверки
print("\nРост выручки по всем компаниям (в рублях):")
for i, (name, growth) in enumerate(zip(names, rev_growth), 1):
    print(f"  {i}. {name}: {growth:,.0f}")

# 3. Находим индексы компаний, отсортированные по росту выручки (по возрастанию)
# argsort возвращает индексы, которые отсортировали бы массив
sorted_indices = np.argsort(rev_growth)

# 4. Берём последние 3 индекса (это компании с наибольшим ростом)
# [::-1] — переворачиваем порядок, чтобы сначала шли самые большие значения
top3_indices = sorted_indices[-3:][::-1]

# 5. Выводим топ-3 компании с наибольшим ростом
print("\n" + "=" * 60)
print("🏆 Топ-3 компании по росту выручки:")
print("=" * 60)
for rank, idx in enumerate(top3_indices, 1):
    print(f"  {rank}. {names[idx]}")
    print(f"     Рост: {rev_growth[idx]:,.0f} руб. "
          f"({revenue[idx, 0]:,.0f} → {revenue[idx, 1]:,.0f})")

Задание 2: Рост выручки и топ-3 компании

Рост выручки по всем компаниям (в рублях):
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 8,816,438
  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ": 164,763,502
  3.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": nan
  4. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ": -120,562,261
  5.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": nan
  6. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "ГРУППА КОМПАНИЙ "САМОЛЕТ": 3,632,216
  7. АКЦИОНЕРНОЕ ОБЩЕСТВО "СЛАВМО": nan
  8. АКЦИОНЕРНОЕ ОБЩЕСТВО "СПЕЦИАЛИЗИРОВАННЫЙ ЗАСТРОЙЩИК "СТРОИТЕЛЬНАЯ КОМПАНИЯ "ВЕК": 674,847
  9. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1": 2,873,920
  10. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 8,816,438

🏆 Топ-3 компании по росту выручки:
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "СЛАВМО"
     Рост: nan руб. (nan → nan)
  2.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕН

### Задание 3
Создайте булеву маску компаний, у которых выручка выросла: `revenue_2023 > revenue_2022`.  
Выведите количество таких компаний и их названия.

In [7]:
# Задание 3: Булева маска роста выручки

print("=" * 60)
print("Задание 3: Компании с ростом выручки")
print("=" * 60)

# 1. Создаём булеву маску: сравниваем выручку 2023 (индекс 1) с 2022 (индекс 0)
# Результат — массив из True/False для каждой компании
revenue_growth_mask = revenue[:, 1] > revenue[:, 0]

# 2. Выводим саму маску для понимания
print("\nБулева маска (True = рост, False = нет роста):")
print(revenue_growth_mask)

# 3. Подсчитываем количество компаний с ростом
# np.sum() для булева массива считает количество True
growth_count = np.sum(revenue_growth_mask)

# 4. Используем маску для индексации списка названий
# Маска выбирает только те элементы, где стоит True
companies_with_growth = [names[i] for i in range(len(names)) if revenue_growth_mask[i]]

# 5. Выводим итоговую статистику
print("\n" + "=" * 60)
print(f"📈 Количество компаний с ростом выручки: {growth_count} из {len(names)}")
print(f"   Доля растущих компаний: {growth_count / len(names) * 100:.1f}%")
print("=" * 60)

# 6. Выводим названия компаний с ростом
print("\nКомпании с ростом выручки:")
for i, name in enumerate(companies_with_growth, 1):
    print(f"  {i}. {name}")

Задание 3: Компании с ростом выручки

Булева маска (True = рост, False = нет роста):
[ True  True False False False  True False  True  True  True]

📈 Количество компаний с ростом выручки: 6 из 10
   Доля растущих компаний: 60.0%

Компании с ростом выручки:
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"
  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ"
  3. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "ГРУППА КОМПАНИЙ "САМОЛЕТ"
  4. АКЦИОНЕРНОЕ ОБЩЕСТВО "СПЕЦИАЛИЗИРОВАННЫЙ ЗАСТРОЙЩИК "СТРОИТЕЛЬНАЯ КОМПАНИЯ "ВЕК"
  5. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1"
  6. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"


### Задание 4
Сформируйте маску «рост выручки И положительная чистая прибыль в 2023».  
Выведите компании, которые удовлетворяют обоим условиям.

In [8]:
# Задание 4: Сложная булева маска (рост выручки И положительная прибыль)

print("=" * 60)
print("Задание 4: Компании с ростом выручки и положительной прибылью")
print("=" * 60)

# 1. Создаём первую маску: выручка выросла (2023 > 2022)
revenue_growth_mask = revenue[:, 1] > revenue[:, 0]

# 2. Создаём вторую маску: чистая прибыль в 2023 положительная (> 0)
# net_profit[:, 1] — столбец 1 это 2023 год
positive_profit_mask = net_profit[:, 1] > 0

# 3. Объединяем маски с помощью логического И (&)
# Важно: каждое условие должно быть в скобках!
combined_mask = (revenue_growth_mask) & (positive_profit_mask)

# 4. Выводим промежуточные результаты для понимания
print("\nМаска роста выручки:")
print(revenue_growth_mask)
print("\nМаска положительной прибыли (2023):")
print(positive_profit_mask)
print("\nОбъединённая маска (И):")
print(combined_mask)

# 5. Подсчитываем количество компаний, удовлетворяющих обоим условиям
sustainable_count = np.sum(combined_mask)

# 6. Отбираем названия компаний по маске
sustainable_companies = [names[i] for i in range(len(names)) if combined_mask[i]]

# 7. Выводим итоговую статистику
print("\n" + "=" * 60)
print(f"✅ Количество компаний, удовлетворяющих обоим условиям: {sustainable_count} из {len(names)}")
print(f"   Доля: {sustainable_count / len(names) * 100:.1f}%")
print("=" * 60)

# 8. Выводим список компаний с деталями
print("\nКомпании с ростом выручки и положительной прибылью:")
for i, idx in enumerate([i for i in range(len(names)) if combined_mask[i]], 1):
    print(f"  {i}. {names[idx]}")
    print(f"     Выручка: {revenue[idx, 0]:,.0f} → {revenue[idx, 1]:,.0f} "
          f"(рост: {revenue[idx, 1] - revenue[idx, 0]:,.0f})")
    print(f"     Прибыль 2023: {net_profit[idx, 1]:,.0f}")

Задание 4: Компании с ростом выручки и положительной прибылью

Маска роста выручки:
[ True  True False False False  True False  True  True  True]

Маска положительной прибыли (2023):
[ True False False  True False  True False  True False  True]

Объединённая маска (И):
[ True False False False False  True False  True False  True]

✅ Количество компаний, удовлетворяющих обоим условиям: 4 из 10
   Доля: 40.0%

Компании с ростом выручки и положительной прибылью:
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"
     Выручка: 1,851,540 → 10,667,978 (рост: 8,816,438)
     Прибыль 2023: 91,493,932
  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "ГРУППА КОМПАНИЙ "САМОЛЕТ"
     Выручка: 3,209,771 → 6,841,987 (рост: 3,632,216)
     Прибыль 2023: 7,667,268
  3. АКЦИОНЕРНОЕ ОБЩЕСТВО "СПЕЦИАЛИЗИРОВАННЫЙ ЗАСТРОЙЩИК "СТРОИТЕЛЬНАЯ КОМПАНИЯ "ВЕК"
     Выручка: 644,115 → 1,318,962 (рост: 674,847)
     Прибыль 2023: 870,349
  4. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"
     Выручка: 

### Задание 5
Рассчитайте маржу: `margin = net_profit / revenue` для обоих лет.  
Используйте `np.divide` и `where`, чтобы избежать деления на 0.  
Выведите среднюю маржу по рынку в 2022 и 2023.

In [11]:
# Задание 5: Расчёт маржи с защитой от деления на 0

print("=" * 60)
print("Задание 5: Расчёт маржи (чистая прибыль / выручка)")
print("=" * 60)

# 1. Создаём маску валидных данных
# Делить можно только там, где выручка не 0 и не NaN, и прибыль не NaN
valid_mask = (revenue != 0) & (~np.isnan(revenue)) & (~np.isnan(net_profit))

# 2. Рассчитываем маржу через np.where
# Если условие истинно — делим, иначе — ставим NaN
margin = np.where(
    valid_mask,                    # условие
    net_profit / revenue,          # если True: считаем маржу
    np.nan                         # если False: ставим пропуск
)

# 3. Выводим маржу для проверки (первые 5 компаний)
print("\nМаржа по компаниям (2022 и 2023):")
print("Формат: [маржа_2022, маржа_2023]")
for i, name in enumerate(names[:5], 1):
    print(f"  {i}. {name}: [{margin[i, 0]:.2%}, {margin[i, 1]:.2%}]")

# 4. Рассчитываем среднюю маржу по рынку, игнорируя NaN
# np.nanmean вычисляет среднее, пропуская значения NaN
avg_margin_2022 = np.nanmean(margin[:, 0])
avg_margin_2023 = np.nanmean(margin[:, 1])

# 5. Выводим итоговую статистику
print("\n" + "=" * 60)
print("📊 Средняя маржа по рынку:")
print("=" * 60)
print(f"  2022 год: {avg_margin_2022:.2%}")
print(f"  2023 год: {avg_margin_2023:.2%}")

# 6. Дополнительная информация: изменение средней маржи
margin_change = avg_margin_2023 - avg_margin_2022
print(f"\n  Изменение: {margin_change:+.2%} "
      f"({'↑ рост' if margin_change > 0 else '↓ падение'})")

Задание 5: Расчёт маржи (чистая прибыль / выручка)

Маржа по компаниям (2022 и 2023):
Формат: [маржа_2022, маржа_2023]
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": [nan%, nan%]
  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ": [nan%, nan%]
  3.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": [27.49%, 23.80%]
  4. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ": [nan%, nan%]
  5.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": [232.91%, 112.06%]

📊 Средняя маржа по рынку:
  2022 год: 89.85%
  2023 год: 383.43%

  Изменение: +293.58% (↑ рост)


### Задание 6
Примените ufunc `np.log1p` к выручке 2023 года.  
Выведите min/max результата (игнорируя NaN).

In [12]:
# Задание 6: Применение np.log1p к выручке 2023 года

print("=" * 60)
print("Задание 6: Логарифмическое преобразование выручки (2023)")
print("=" * 60)

# 1. Берём выручку за 2023 год (столбец с индексом 1)
revenue_2023 = revenue[:, 1]

# 2. Выводим исходные данные для понимания
print("\nИсходная выручка за 2023 год:")
for i, (name, val) in enumerate(zip(names, revenue_2023), 1):
    print(f"  {i}. {name}: {val:,.0f}" if not np.isnan(val) else f"  {i}. {name}: NaN")

# 3. Применяем ufunc np.log1p: вычисляет log(1 + x) для каждого элемента
# Это полезно для работы с большими числами и уменьшения влияния выбросов
# log1p более точен для малых значений, чем просто log(1+x)
log_revenue_2023 = np.log1p(revenue_2023)

# 4. Выводим преобразованные значения для проверки
print("\nПосле применения np.log1p:")
for i, (name, val) in enumerate(zip(names, log_revenue_2023), 1):
    print(f"  {i}. {name}: {val:.2f}" if not np.isnan(val) else f"  {i}. {name}: NaN")

# 5. Находим минимальное и максимальное значение, игнорируя NaN
# np.nanmin / np.nanmax пропускают пропущенные значения
min_log = np.nanmin(log_revenue_2023)
max_log = np.nanmax(log_revenue_2023)

# 6. Выводим итоговую статистику
print("\n" + "=" * 60)
print("📊 Статистика после логарифмирования:")
print("=" * 60)
print(f"  Минимальное значение: {min_log:.2f}")
print(f"  Максимальное значение: {max_log:.2f}")
print(f"  Диапазон (размах): {max_log - min_log:.2f}")

# 7. Дополнительная информация: обратное преобразование для понимания
# np.expm1(x) — обратная функция к log1p: exp(x) - 1
print("\n🔍 Интерпретация:")
print(f"  min: log1p(x) = {min_log:.2f} → x ≈ {np.expm1(min_log):,.0f} руб.")
print(f"  max: log1p(x) = {max_log:.2f} → x ≈ {np.expm1(max_log):,.0f} руб.")

Задание 6: Логарифмическое преобразование выручки (2023)

Исходная выручка за 2023 год:
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 10,667,978
  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ": 497,511,315
  3.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": NaN
  4. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ": 2,753,475,003
  5.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": NaN
  6. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "ГРУППА КОМПАНИЙ "САМОЛЕТ": 6,841,987
  7. АКЦИОНЕРНОЕ ОБЩЕСТВО "СЛАВМО": NaN
  8. АКЦИОНЕРНОЕ ОБЩЕСТВО "СПЕЦИАЛИЗИРОВАННЫЙ ЗАСТРОЙЩИК "СТРОИТЕЛЬНАЯ КОМПАНИЯ "ВЕК": 1,318,962
  9. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1": 101,697,792
  10. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 10,667,978

После применения np.log1p:
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 16.18
  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ

### Задание 7
Сравните компании по капиталу 2023: найдите топ-5 по `equity[:,1]`.  
Используйте `argsort` и индексацию.

In [14]:
# Задание 7: Топ-5 компаний по капиталу (2023)

print("=" * 60)
print("Задание 7: Топ-5 компаний по капиталу (2023)")
print("=" * 60)

# 1. Берём капитал за 2023 год (столбец с индексом 1)
equity_2023 = equity[:, 1]

# 2. Создаём маску валидных данных (не NaN)
# ~np.isnan() означает "НЕ является NaN"
valid_mask = ~np.isnan(equity_2023)

# 3. Выводим исходные данные для понимания
print("\nКапитал всех компаний за 2023 год:")
for i, (name, val) in enumerate(zip(names, equity_2023), 1):
    if np.isnan(val):
        print(f"  {i}. {name}: NaN (нет данных)")
    else:
        print(f"  {i}. {name}: {val:,.0f}")

# 4. Получаем индексы только валидных компаний
valid_indices = np.where(valid_mask)[0]

# 5. Если валидных компаний меньше 5, корректируем число
top_n = min(5, len(valid_indices))

if top_n == 0:
    print("\n⚠️ Нет данных о капитале ни для одной компании!")
else:
    # 6. Сортируем валидные компании по капиталу (по возрастанию)
    # Берём значения капитала только для валидных индексов
    valid_equity = equity_2023[valid_indices]
    sorted_valid_indices = np.argsort(valid_equity)

    # 7. Берём последние top_n индексов (самые большие значения)
    # и переворачиваем порядок, чтобы сначала шли самые крупные
    top_indices = sorted_valid_indices[-top_n:][::-1]

    # 8. Преобразуем локальные индексы в глобальные (оригинальные)
    top_global_indices = valid_indices[top_indices]

    # 9. Выводим топ-5 компаний с деталями
    print("\n" + "=" * 60)
    print(f"🏆 Топ-{top_n} компаний по капиталу (2023):")
    print("=" * 60)
    for rank, idx in enumerate(top_global_indices, 1):
        print(f"  {rank}. {names[idx]}")

        # Проверяем, есть ли данные для форматирования
        if np.isnan(equity[idx, 1]):
            print(f"     Капитал 2023: нет данных")
            print(f"     Капитал 2022: нет данных")
            print(f"     Изменение: нет данных")
        else:
            # ИСПРАВЛЕНИЕ: правильный формат specifier :+,.0f
            cap_2023 = equity[idx, 1]
            cap_2022 = equity[idx, 0]
            change = cap_2023 - cap_2022

            print(f"     Капитал 2023: {cap_2023:,.0f} руб.")
            print(f"     Капитал 2022: {cap_2022:,.0f} руб.")
            print(f"     Изменение: {change:+,.0f} руб.")
        print()

Задание 7: Топ-5 компаний по капиталу (2023)

Капитал всех компаний за 2023 год:
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 1,516,195,460
  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ": NaN (нет данных)
  3.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": NaN (нет данных)
  4. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ": 1,308,113,431
  5.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": NaN (нет данных)
  6. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "ГРУППА КОМПАНИЙ "САМОЛЕТ": 17,473,552
  7. АКЦИОНЕРНОЕ ОБЩЕСТВО "СЛАВМО": NaN (нет данных)
  8. АКЦИОНЕРНОЕ ОБЩЕСТВО "СПЕЦИАЛИЗИРОВАННЫЙ ЗАСТРОЙЩИК "СТРОИТЕЛЬНАЯ КОМПАНИЯ "ВЕК": 5,115,194
  9. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1": 142,046,969
  10. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 1,516,195,460

🏆 Топ-5 компаний по капиталу (2023):
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС

### Задание 8
Постройте маску «устойчивые»: `equity_2023 / assets_2023 >= 0.3`.  
Выведите долю таких компаний.

In [15]:
# Задание 8: Маска «устойчивые» компании (капитал / активы >= 0.3)

print("=" * 60)
print("Задание 8: Финансовая устойчивость компаний (2023)")
print("=" * 60)

# 1. Берём капитал и активы за 2023 год (столбец с индексом 1)
equity_2023 = equity[:, 1]
assets_2023 = assets[:, 1]

# 2. Создаём маску валидных данных: оба значения должны быть не NaN
valid_mask = (~np.isnan(equity_2023)) & (~np.isnan(assets_2023)) & (assets_2023 != 0)

# 3. Рассчитываем коэффициент финансовой независимости: капитал / активы
# Используем np.where для безопасного деления только по валидным данным
equity_ratio = np.where(
    valid_mask,                    # условие: данные валидны
    equity_2023 / assets_2023,     # если True: считаем отношение
    np.nan                         # если False: ставим пропуск
)

# 4. Создаём маску «устойчивых»: отношение >= 0.3 (30%)
stable_mask = equity_ratio >= 0.3

# 5. Выводим коэффициент для каждой компании (для понимания)
print("\nКоэффициент финансовой независимости (капитал / активы):")
for i, (name, ratio) in enumerate(zip(names, equity_ratio), 1):
    if np.isnan(ratio):
        print(f"  {i}. {name}: нет данных")
    else:
        status = "✅ устойчивая" if stable_mask[i-1] else "❌ неустойчивая"
        print(f"  {i}. {name}: {ratio:.2%} {status}")

# 6. Подсчитываем количество устойчивых компаний (только среди тех, у кого есть данные)
# Сначала считаем, у скольких компаний есть данные
companies_with_data = np.sum(valid_mask)
stable_count = np.sum(stable_mask & valid_mask)  # устойчивые среди валидных

# 7. Рассчитываем долю устойчивых компаний
if companies_with_data > 0:
    stable_proportion = stable_count / companies_with_data
else:
    stable_proportion = np.nan

# 8. Выводим итоговую статистику
print("\n" + "=" * 60)
print("📊 Итоги по финансовой устойчивости:")
print("=" * 60)
print(f"  Компаний с данными: {companies_with_data} из {len(names)}")
print(f"  Устойчивых компаний (капитал ≥ 30% активов): {stable_count}")
print(f"  Доля устойчивых: {stable_proportion:.1%}" if not np.isnan(stable_proportion) else "  Доля устойчивых: нет данных")
print(f"  Порог устойчивости: 30% (0.3)")
print("=" * 60)

# 9. Выводим список устойчивых компаний
if stable_count > 0:
    print("\n✅ Список устойчивых компаний:")
    for i, (name, ratio) in enumerate(zip(names, equity_ratio), 1):
        if not np.isnan(ratio) and ratio >= 0.3:
            print(f"  • {name} ({ratio:.2%})")

Задание 8: Финансовая устойчивость компаний (2023)

Коэффициент финансовой независимости (капитал / активы):
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 66.62% ✅ устойчивая
  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ": нет данных
  3.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": нет данных
  4. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ": 48.26% ✅ устойчивая
  5.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": нет данных
  6. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "ГРУППА КОМПАНИЙ "САМОЛЕТ": 9.18% ❌ неустойчивая
  7. АКЦИОНЕРНОЕ ОБЩЕСТВО "СЛАВМО": нет данных
  8. АКЦИОНЕРНОЕ ОБЩЕСТВО "СПЕЦИАЛИЗИРОВАННЫЙ ЗАСТРОЙЩИК "СТРОИТЕЛЬНАЯ КОМПАНИЯ "ВЕК": 77.41% ✅ устойчивая
  9. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1": 72.25% ✅ устойчивая
  10. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 66.62% ✅ устойчивая

📊 Итоги по финансовой устойчивости:
  Компаний с 

### Задание 9
Категоризуйте компании по росту выручки: рост > 0 → 1, иначе → 0.  
Используйте `np.where`. Выведите массив категорий и частоты.

In [16]:
# Задание 9: Категоризация компаний по росту выручки

print("=" * 60)
print("Задание 9: Категоризация по росту выручки")
print("=" * 60)

# 1. Рассчитываем рост выручки: 2023 год (индекс 1) минус 2022 год (индекс 0)
rev_growth = revenue[:, 1] - revenue[:, 0]

# 2. Выводим рост для понимания
print("\nРост выручки по компаниям:")
for i, (name, growth) in enumerate(zip(names, rev_growth), 1):
    if np.isnan(growth):
        print(f"  {i}. {name}: нет данных")
    else:
        print(f"  {i}. {name}: {growth:,.0f}")

# 3. Создаём массив категорий с помощью np.where
# Если рост > 0 → 1 (компания растёт), иначе → 0 (нет роста или убыток)
# np.nan_to_num заменяет NaN на 0 перед сравнением, чтобы избежать ошибок
categories = np.where(
    np.nan_to_num(rev_growth, nan=-np.inf) > 0,  # условие
    1,                                            # если True
    0                                             # если False
)

# 4. Выводим массив категорий
print("\n" + "=" * 60)
print("Массив категорий (1 = рост, 0 = нет роста):")
print("=" * 60)
print(categories)

# 5. Выводим категории с названиями компаний
print("\nКатегории по компаниям:")
for i, (name, cat) in enumerate(zip(names, categories), 1):
    status = "📈 растёт" if cat == 1 else "📉 нет роста"
    print(f"  {i}. {name}: {cat} {status}")

# 6. Рассчитываем частоты (количество компаний в каждой категории)
# np.bincount подсчитывает количество вхождений каждого значения (0, 1, 2, ...)
frequencies = np.bincount(categories)

# 7. Выводим частоты
print("\n" + "=" * 60)
print("📊 Частоты категорий:")
print("=" * 60)
print(f"  Категория 0 (нет роста): {frequencies[0]} компаний")
print(f"  Категория 1 (рост): {frequencies[1]} компаний")
print(f"  Всего компаний: {np.sum(frequencies)}")

# 8. Дополнительная статистика: доля растущих компаний
if len(categories) > 0:
    growth_share = frequencies[1] / len(categories)
    print(f"\n  Доля растущих компаний: {growth_share:.1%}")

Задание 9: Категоризация по росту выручки

Рост выручки по компаниям:
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 8,816,438
  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ": 164,763,502
  3.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": нет данных
  4. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ": -120,562,261
  5.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": нет данных
  6. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "ГРУППА КОМПАНИЙ "САМОЛЕТ": 3,632,216
  7. АКЦИОНЕРНОЕ ОБЩЕСТВО "СЛАВМО": нет данных
  8. АКЦИОНЕРНОЕ ОБЩЕСТВО "СПЕЦИАЛИЗИРОВАННЫЙ ЗАСТРОЙЩИК "СТРОИТЕЛЬНАЯ КОМПАНИЯ "ВЕК": 674,847
  9. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1": 2,873,920
  10. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 8,816,438

Массив категорий (1 = рост, 0 = нет роста):
[1 1 0 0 0 1 0 1 1 1]

Категории по компаниям:
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КО

### Задание 10
Создайте матрицу признаков X размера (n,4) для 2023 года:  
[выручка, чистая прибыль, активы, капитал]. Проверьте форму и выведите первые 3 строки.

In [17]:
# Задание 10: Создание матрицы признаков X (n, 4) для 2023 года

print("=" * 60)
print("Задание 10: Матрица признаков X (2023 год)")
print("=" * 60)

# 1. Извлекаем данные за 2023 год (столбец с индексом 1)
# revenue[:, 1] — все строки, столбец 1 (2023 год)
revenue_2023 = revenue[:, 1]
net_profit_2023 = net_profit[:, 1]
assets_2023 = assets[:, 1]
equity_2023 = equity[:, 1]

# 2. Создаём матрицу признаков X с помощью np.column_stack
# np.column_stack объединяет несколько 1D-массивов в 2D-матрицу по столбцам
X = np.column_stack([
    revenue_2023,      # столбец 0: выручка
    net_profit_2023,   # столбец 1: чистая прибыль
    assets_2023,       # столбец 2: активы
    equity_2023        # столбец 3: капитал
])

# 3. Проверяем форму матрицы
print("\n📊 Форма матрицы X:")
print(f"  X.shape = {X.shape}")
print(f"  Ожидается: (10 компаний, 4 признака)")

# 4. Выводим первые 3 строки для проверки
print("\n📋 Первые 3 строки матрицы X:")
print("Формат: [выручка, чистая прибыль, активы, капитал]")
for i in range(min(3, len(X))):
    print(f"  Компания {i+1}:")
    for j, val in enumerate(X[i]):
        if np.isnan(val):
            print(f"    [{j}] {['выручка', 'прибыль', 'активы', 'капитал'][j]}: NaN")
        else:
            print(f"    [{j}] {['выручка', 'прибыль', 'активы', 'капитал'][j]}: {val:,.0f}")

# 5. Альтернативный вывод через slicing (более компактный)
print("\n" + "=" * 60)
print("Матрица X (первые 3 строки, все столбцы):")
print("=" * 60)
print(X[:3])

# 6. Дополнительная статистика по матрице
print("\n" + "=" * 60)
print("📈 Статистика по признакам (2023):")
print("=" * 60)
feature_names = ['Выручка', 'Чистая прибыль', 'Активы', 'Капитал']
for i, name in enumerate(feature_names):
    col = X[:, i]
    valid_count = np.sum(~np.isnan(col))
    print(f"  {name}:")
    print(f"    Компания с данными: {valid_count} из {len(col)}")
    if valid_count > 0:
        print(f"    Среднее: {np.nanmean(col):,.0f}")
        print(f"    Мин: {np.nanmin(col):,.0f}")
        print(f"    Макс: {np.nanmax(col):,.0f}")

Задание 10: Матрица признаков X (2023 год)

📊 Форма матрицы X:
  X.shape = (10, 4)
  Ожидается: (10 компаний, 4 признака)

📋 Первые 3 строки матрицы X:
Формат: [выручка, чистая прибыль, активы, капитал]
  Компания 1:
    [0] выручка: 10,667,978
    [1] прибыль: 91,493,932
    [2] активы: 2,275,887,506
    [3] капитал: 1,516,195,460
  Компания 2:
    [0] выручка: 497,511,315
    [1] прибыль: NaN
    [2] активы: 934,268,135
    [3] капитал: NaN
  Компания 3:
    [0] выручка: NaN
    [1] прибыль: NaN
    [2] активы: NaN
    [3] капитал: NaN

Матрица X (первые 3 строки, все столбцы):
[[1.06679780e+07 9.14939320e+07 2.27588751e+09 1.51619546e+09]
 [4.97511315e+08            nan 9.34268135e+08            nan]
 [           nan            nan            nan            nan]]

📈 Статистика по признакам (2023):
  Выручка:
    Компания с данными: 7 из 10
    Среднее: 483,168,716
    Мин: 1,318,962
    Макс: 2,753,475,003
  Чистая прибыль:
    Компания с данными: 5 из 10
    Среднее: 169,362,987
  

### Задание 11
Найдите корреляцию (Pearson) между выручкой и активами (2023) через `np.corrcoef`.  
Выведите одно число corr (предварительно удалив NaN).

In [18]:
# Задание 11: Корреляция Пирсона между выручкой и активами (2023)

print("=" * 60)
print("Задание 11: Корреляция выручки и активов (2023)")
print("=" * 60)

# 1. Извлекаем данные за 2023 год (столбец с индексом 1)
revenue_2023 = revenue[:, 1]
assets_2023 = assets[:, 1]

# 2. Создаём маску валидных пар: оба значения должны быть не NaN
# Это важно: удаляем строки, где хотя бы одно значение пропущено
valid_mask = (~np.isnan(revenue_2023)) & (~np.isnan(assets_2023))

# 3. Фильтруем данные, оставляя только валидные пары
revenue_valid = revenue_2023[valid_mask]
assets_valid = assets_2023[valid_mask]

# 4. Выводим информацию о данных
print(f"\n📊 Данные для расчёта:")
print(f"  Всего компаний: {len(revenue_2023)}")
print(f"  Компаний с полными данными: {len(revenue_valid)}")

# 5. Проверяем, достаточно ли данных для расчёта
if len(revenue_valid) < 2:
    print("\n⚠️ Недостаточно данных для расчёта корреляции (нужно минимум 2 пары)!")
    corr = np.nan
else:
    # 6. Рассчитываем матрицу корреляции с помощью np.corrcoef
    # Функция возвращает матрицу 2×2:
    # [[1.0,   corr],
    #  [corr,  1.0]]
    corr_matrix = np.corrcoef(revenue_valid, assets_valid)

    # 7. Извлекаем коэффициент корреляции (элемент [0,1] или [1,0])
    corr = corr_matrix[0, 1]

# 8. Выводим результат
print("\n" + "=" * 60)
print("📈 Результат:")
print("=" * 60)
if np.isnan(corr):
    print("  Корреляция: не удалось рассчитать (нет данных)")
else:
    print(f"  Коэффициент корреляции Пирсона: {corr:.4f}")

    # 9. Интерпретируем результат
    if corr > 0.9:
        interpretation = "🔗 Очень сильная прямая связь"
    elif corr > 0.7:
        interpretation = "🔗 Сильная прямая связь"
    elif corr > 0.5:
        interpretation = "🔗 Умеренная прямая связь"
    elif corr > 0.3:
        interpretation = "🔗 Слабая прямая связь"
    elif corr > -0.3:
        interpretation = "⚪ Практически нет связи"
    elif corr > -0.5:
        interpretation = "🔗 Слабая обратная связь"
    elif corr > -0.7:
        interpretation = "🔗 Умеренная обратная связь"
    elif corr > -0.9:
        interpretation = "🔗 Сильная обратная связь"
    else:
        interpretation = "🔗 Очень сильная обратная связь"

    print(f"  Интерпретация: {interpretation}")

# 10. Дополнительная визуализация (опционально)
if not np.isnan(corr) and len(revenue_valid) >= 2:
    print("\n🔍 Проверка данных:")
    print(f"  Выручка: мин={np.min(revenue_valid):,.0f}, макс={np.max(revenue_valid):,.0f}")
    print(f"  Активы:  мин={np.min(assets_valid):,.0f}, макс={np.max(assets_valid):,.0f}")

Задание 11: Корреляция выручки и активов (2023)

📊 Данные для расчёта:
  Всего компаний: 10
  Компаний с полными данными: 7

📈 Результат:
  Коэффициент корреляции Пирсона: 0.5423
  Интерпретация: 🔗 Умеренная прямая связь

🔍 Проверка данных:
  Выручка: мин=1,318,962, макс=2,753,475,003
  Активы:  мин=6,607,975, макс=2,710,708,856


### Задание 12
Отсортируйте компании по марже 2023 (`margin[:,1]`) по убыванию и выведите топ-5.  
Учтите NaN: `np.nan_to_num(..., nan=-inf)` для корректной сортировки.

In [19]:
# Задание 12: Топ-5 компаний по марже 2023 (с обработкой NaN)

print("=" * 60)
print("Задание 12: Топ-5 компаний по марже (2023)")
print("=" * 60)

# 1. Берём маржу за 2023 год (столбец с индексом 1)
margin_2023 = margin[:, 1]

# 2. Выводим исходные данные для понимания
print("\nМаржа всех компаний за 2023 год:")
for i, (name, val) in enumerate(zip(names, margin_2023), 1):
    if np.isnan(val):
        print(f"  {i}. {name}: NaN (нет данных)")
    else:
        print(f"  {i}. {name}: {val:.2%}")

# 3. Обрабатываем NaN для корректной сортировки
# Заменяем NaN на -inf, чтобы такие компании оказались в конце при сортировке по убыванию
margin_for_sort = np.nan_to_num(margin_2023, nan=-np.inf)

# 4. Получаем индексы, которые отсортировали бы массив по возрастанию
sorted_indices = np.argsort(margin_for_sort)

# 5. Берём последние 5 индексов (самые большие значения)
# и переворачиваем порядок, чтобы сначала шли самые высокие
top5_indices = sorted_indices[-5:][::-1]

# 6. Выводим топ-5 компаний с деталями
print("\n" + "=" * 60)
print("🏆 Топ-5 компаний по марже (2023):")
print("=" * 60)
for rank, idx in enumerate(top5_indices, 1):
    margin_val = margin_2023[idx]

    print(f"  {rank}. {names[idx]}")

    if np.isnan(margin_val):
        print(f"     Маржа 2023: нет данных")
        print(f"     Маржа 2022: нет данных")
    else:
        print(f"     Маржа 2023: {margin_val:.2%}")
        print(f"     Маржа 2022: {margin[:, 0][idx]:.2%}")
        print(f"     Изменение: {margin_val - margin[:, 0][idx]:+.2%}")
    print()

# 7. Дополнительная статистика
print("=" * 60)
print("📊 Статистика по марже (2023):")
print("=" * 60)
valid_margins = margin_2023[~np.isnan(margin_2023)]
if len(valid_margins) > 0:
    print(f"  Компаний с данными: {len(valid_margins)} из {len(margin_2023)}")
    print(f"  Средняя маржа: {np.mean(valid_margins):.2%}")
    print(f"  Медианная маржа: {np.median(valid_margins):.2%}")
    print(f"  Мин маржа: {np.min(valid_margins):.2%}")
    print(f"  Макс маржа: {np.max(valid_margins):.2%}")

Задание 12: Топ-5 компаний по марже (2023)

Маржа всех компаний за 2023 год:
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 857.65%
  2. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "АЭРОФЛОТ-РОССИЙСКИЕ АВИАЛИНИИ": NaN (нет данных)
  3.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": NaN (нет данных)
  4. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "НЕФТЯНАЯ КОМПАНИЯ "ЛУКОЙЛ": 23.80%
  5.  АКЦИОНЕРНОЕ ОБЩЕСТВО "ГАЗПРОМ ГАЗОРАСПРЕДЕЛЕНИЕ ПЕТРОЗАВОДСК": NaN (нет данных)
  6. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "ГРУППА КОМПАНИЙ "САМОЛЕТ": 112.06%
  7. АКЦИОНЕРНОЕ ОБЩЕСТВО "СЛАВМО": NaN (нет данных)
  8. АКЦИОНЕРНОЕ ОБЩЕСТВО "СПЕЦИАЛИЗИРОВАННЫЙ ЗАСТРОЙЩИК "СТРОИТЕЛЬНАЯ КОМПАНИЯ "ВЕК": 65.99%
  9. ПУБЛИЧНОЕ АКЦИОНЕРНОЕ  ОБЩЕСТВО  "ТЕРРИТОРИАЛЬНАЯ  ГЕНЕРИРУЮЩАЯ  КОМПАНИЯ   №1": NaN (нет данных)
  10. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС": 857.65%

🏆 Топ-5 компаний по марже (2023):
  1. АКЦИОНЕРНОЕ ОБЩЕСТВО "АТОМНЫЙ ЭНЕРГОПРОМЫШЛЕННЫЙ КОМПЛЕКС"
     Маржа 2023: 857.65%
